# FruitFly Brain API (Google Colab)
Runs a spiking fly-brain on the **real connectome** (NeuPrint) and serves it as an API.
Your game page (`fruitfly_colab_game.html`) sends "how far is the next hazard" and gets back "jump or not".

**Steps**
1. Get a free NeuPrint token: open https://neuprint.janelia.org, log in with Google, click your account icon -> *Auth Token*.
2. (Optional) In Colab left bar click the key icon (Secrets), add `NEUPRINT_TOKEN`, turn on notebook access. Otherwise you will be asked to paste it.
3. Run cells 1 -> 4 in order (Runtime -> Run all). Cell 4 prints an **API URL**.
4. Open `fruitfly_colab_game.html` on your PC, paste the URL, press **Connect**.

No token = small demo network (315 neurons) so you can test the pipeline first. `MODE="circuit"` loads the eye -> giant-fiber circuit (up to 20,000 neurons);
`MODE="full"` loads every neuron with >=10 input and output synapses (slow, needs lots of RAM).

In [ ]:
#@title 1) Install
!pip -q install flask flask-cors neuprint-python

In [ ]:
#@title 2) Settings + load the fly connectome (real data via NeuPrint API)
import os, time, getpass, numpy as np, scipy.sparse as sp

DATASET = "male-cns:v1.0"        #@param {type:"string"}
MODE = "circuit"                 #@param ["circuit", "full"]
MAX_NEURONS = 20000              #@param {type:"integer"}
MIN_PARTNER_WEIGHT = 5           #@param {type:"integer"}
VIS_TYPES = ["LC4", "LPLC2"]     # eye neurons that get the "looming" input
DN_TYPES = ["DNp01"]             # giant fiber: when it fires, the fly jumps
CACHE = f"/content/connectome_{MODE}"

class Connectome:
    """W[post, pre] = synapse count (CSR sparse), types, body ids."""
    def __init__(self, W, types, ids):
        self.W, self.types, self.ids = sp.csr_matrix(W, dtype=np.float32), np.array(types).astype(str), np.array(ids)
    @property
    def n(self): return len(self.types)
    def idx(self, *names): return np.where(np.isin(self.types, names))[0]

def get_token():
    try:
        from google.colab import userdata
        t = userdata.get("NEUPRINT_TOKEN")
        if t: return t
    except Exception:
        pass
    t = os.environ.get("NEUPRINT_TOKEN", "")
    if t or os.environ.get("FLY_NO_PROMPT"): return t
    return getpass.getpass("NeuPrint token (leave empty = small DEMO network): ").strip()

def load_demo(n_vis=150, n_inter=150, n_dn=15, seed=42):
    rng = np.random.default_rng(seed)
    types = ["LC4"] * (n_vis // 2) + ["LPLC2"] * (n_vis - n_vis // 2) + ["INTER"] * n_inter + ["DNp01"] * n_dn
    n = len(types); W = np.zeros((n, n), np.float32)
    vis = np.arange(n_vis); inter = np.arange(n_vis, n_vis + n_inter); dn = np.arange(n_vis + n_inter, n)
    def connect(pre, post, p, scale):
        m = rng.random((len(post), len(pre))) < p
        W[np.ix_(post, pre)] += m * rng.exponential(scale, m.shape)
    connect(vis, inter, 0.15, 5.0); connect(inter, inter, 0.03, 2.0); connect(inter, dn, 0.30, 5.0)
    np.fill_diagonal(W, 0)
    return Connectome(W, types, np.arange(n) + 10000)

def load_neuprint(token):
    from neuprint import Client, fetch_neurons, fetch_adjacencies, fetch_simple_connections, NeuronCriteria as NC
    c = Client("neuprint.janelia.org", dataset=DATASET, token=token)
    print("connected to NeuPrint, dataset:", DATASET)
    if MODE == "full":
        nd, _ = fetch_neurons(NC(min_pre=10, min_post=10), client=c)
        ids = nd["bodyId"].tolist()
        print("full mode: neurons fetched:", len(ids))
    else:
        seeds = fetch_neurons(NC(type=VIS_TYPES + DN_TYPES), client=c)[0]["bodyId"].tolist()
        print("seed neurons:", len(seeds), "(" + ", ".join(VIS_TYPES + DN_TYPES) + ")")
        down = fetch_simple_connections(upstream_criteria=NC(type=VIS_TYPES), min_weight=MIN_PARTNER_WEIGHT, client=c)
        up = fetch_simple_connections(downstream_criteria=NC(type=DN_TYPES), min_weight=MIN_PARTNER_WEIGHT, client=c)
        score = {}
        for b, w in zip(down["bodyId_post"], down["weight"]): score[b] = score.get(b, 0) + w
        for b, w in zip(up["bodyId_pre"], up["weight"]): score[b] = score.get(b, 0) + w
        for b in seeds: score.pop(b, None)
        room = max(0, MAX_NEURONS - len(seeds))
        partners = [b for b, _ in sorted(score.items(), key=lambda kv: -kv[1])[:room]]
        ids = list(dict.fromkeys(seeds + partners))
        print("partners added:", len(partners), "-> total", len(ids))
    nd, conn = fetch_adjacencies(sources=ids, targets=ids, min_total_weight=2, client=c)
    conn = conn.groupby(["bodyId_pre", "bodyId_post"], as_index=False)["weight"].sum()
    nd = nd.drop_duplicates("bodyId").reset_index(drop=True)
    idmap = {b: i for i, b in enumerate(nd["bodyId"].tolist())}
    conn = conn[conn["bodyId_pre"].isin(idmap) & conn["bodyId_post"].isin(idmap)]
    rows = conn["bodyId_post"].map(idmap).to_numpy(); cols = conn["bodyId_pre"].map(idmap).to_numpy()
    W = sp.csr_matrix((conn["weight"].to_numpy(np.float32), (rows, cols)), shape=(len(nd), len(nd)))
    types = nd["type"].fillna("?").astype(str).to_numpy() if "type" in nd else np.array(["?"] * len(nd))
    return Connectome(W, types, nd["bodyId"].to_numpy())

def save_cache(cn):
    sp.save_npz(CACHE + "_W.npz", cn.W); np.savez(CACHE + "_meta.npz", types=cn.types, ids=cn.ids)

def load_cache():
    if os.path.exists(CACHE + "_W.npz") and os.path.exists(CACHE + "_meta.npz"):
        m = np.load(CACHE + "_meta.npz", allow_pickle=True)
        return Connectome(sp.load_npz(CACHE + "_W.npz"), m["types"], m["ids"])

cn = load_cache()
if cn is not None:
    print("loaded from cache")
else:
    TOKEN = get_token()
    if TOKEN:
        t0 = time.time(); cn = load_neuprint(TOKEN); save_cache(cn); print(f"download took {time.time()-t0:.0f}s, cached")
    else:
        cn = load_demo(); print("NO TOKEN -> using DEMO network (not real fly data)")

nvis, ndn = len(cn.idx(*VIS_TYPES)), len(cn.idx(*DN_TYPES))
print(f"neurons={cn.n}  synapse-links={cn.W.nnz}  eye(LC4/LPLC2)={nvis}  giant-fiber(DNp01)={ndn}")
assert nvis > 0 and ndn > 0, "No LC4/LPLC2 or DNp01 neurons found - check DATASET / token."

In [ ]:
#@title 3) Brain (spiking network) + auto-calibration + API server code
import collections, threading, json, time, re, subprocess
import numpy as np, scipy.sparse as sp

class Brain:
    """Leaky integrate-and-fire network on a sparse connectome."""
    def __init__(self, cn, vis_types, dn_types, w_scale=0.5, w_max=8.0, inh_frac=0.15, seed=0):
        self.cn, self.raw = cn, cn.W.tocsr().astype(np.float32)
        self.n = cn.n
        self.vis = cn.idx(*vis_types); self.dn = cn.idx(*dn_types)
        self.vis_mask = np.zeros(self.n, np.float32); self.vis_mask[self.vis] = 1
        rng = np.random.default_rng(seed)
        cand = np.setdiff1d(np.arange(self.n), np.concatenate([self.vis, self.dn]))
        inh = rng.choice(cand, int(len(cand) * inh_frac), replace=False) if len(cand) else []
        self.sign = np.ones(self.n, np.float32); self.sign[inh] = -2.0       # some inhibitory cells
        self.w_max = w_max
        self.dt, self.tau = 0.001, 0.01
        self.v_rest, self.v_th, self.v_reset = -70.0, -50.0, -75.0
        self.set_w_scale(w_scale); self.reset()
    def set_w_scale(self, ws):
        W = self.raw.copy(); W.data = np.minimum(W.data * ws, self.w_max).astype(np.float32)
        self.W = (W @ sp.diags(self.sign)).tocsr().astype(np.float32); self.w_scale = ws
    def reset(self):
        self.v = np.full(self.n, self.v_rest, np.float32); self.prev = np.zeros(self.n, np.float32)
    def step(self, drive):
        syn = self.W.dot(self.prev)
        self.v += (-(self.v - self.v_rest) + drive * self.vis_mask) * (self.dt / self.tau) + syn
        s = self.v >= self.v_th
        self.v[s] = self.v_reset
        self.prev = s.astype(np.float32)
        return s

class FlyController:
    def __init__(self, brain, sense_range=60.0, gain=45.0, substeps=4, window=3, gf_threshold=3, cooldown=18, sample=1200):
        self.b, self.range, self.gain, self.sub = brain, sense_range, gain, substeps
        self.win = collections.deque(maxlen=window); self.gf_th, self.cool_n, self.cool = gf_threshold, cooldown, 0
        rng = np.random.default_rng(1)
        others = np.setdiff1d(np.arange(brain.n), np.concatenate([brain.vis, brain.dn]))
        k_vis = min(len(brain.vis), sample // 3); k_dn = min(len(brain.dn), 40)
        k_oth = min(len(others), sample - k_vis - k_dn)
        self.samp = np.concatenate([rng.choice(brain.vis, k_vis, replace=False), brain.dn[:k_dn],
                                    rng.choice(others, k_oth, replace=False) if k_oth > 0 else np.array([], int)]).astype(int)
    def reset(self):
        self.b.reset(); self.win.clear(); self.cool = 0
    def frame(self, dist, on_ground=True):
        t0 = time.perf_counter()
        danger = 0.0 if dist is None or dist <= -5 else float(np.clip(1 - dist / self.range, 0, 1))
        gf = active = 0; any_s = np.zeros(self.b.n, bool)
        for _ in range(self.sub):
            s = self.b.step(self.gain * danger)
            gf += int(s[self.b.dn].sum()); active += int(s.sum()); any_s |= s
        self.win.append(gf); gf_sum = sum(self.win)
        self.cool = max(0, self.cool - 1)
        jump = bool(gf_sum >= self.gf_th and on_ground and self.cool == 0)
        if jump: self.cool = self.cool_n
        return dict(jump=jump, danger=round(danger, 3), gf_spikes=gf_sum, active=active, neurons=int(self.b.n),
                    latency_ms=round((time.perf_counter() - t0) * 1000, 2),
                    spk=np.nonzero(any_s[self.samp])[0].tolist())

def _first_jump_dist(ctrl, start=120.0, step=3.0, frames=90):
    ctrl.reset(); d = start
    for _ in range(frames):
        if ctrl.frame(d, True)["jump"]: return d
        d -= step
    return None

def calibrate(ctrl, target=22.0, lo=12.0, hi=36.0):
    """Find (w_scale, gain) so the fly jumps when a hazard is ~`target` units ahead and never when nothing is near."""
    best = None; t0 = time.time()
    for ws in [0.15, 0.25, 0.4, 0.6, 1.0, 1.6, 2.5]:
        ctrl.b.set_w_scale(ws)
        for gain in [25, 35, 45, 60, 80, 110]:
            ctrl.gain = gain
            d = _first_jump_dist(ctrl)
            quiet = True; ctrl.reset()
            for _ in range(30):
                if ctrl.frame(None, True)["jump"]: quiet = False; break
            if d is not None and quiet and lo <= d <= hi:
                err = abs(d - target)
                if best is None or err < best[0]: best = (err, ws, gain, d)
    if best is None:
        print("calibration: no setting hit the target window - using defaults (tune w_scale/gain by hand)")
        ctrl.b.set_w_scale(0.5); ctrl.gain = 45.0
    else:
        _, ws, gain, d = best; ctrl.b.set_w_scale(ws); ctrl.gain = float(gain)
        print(f"calibrated: w_scale={ws} gain={gain} -> jumps at ~{d:.0f} units ahead ({time.time()-t0:.0f}s)")
    ctrl.reset(); return best is not None

brain = Brain(cn, VIS_TYPES, DN_TYPES)
ctrl = FlyController(brain)
calibrate(ctrl)

# ---------------------------------------------------------------- Flask API
from flask import Flask, request, jsonify
from flask_cors import CORS
app = Flask(__name__); CORS(app)
_lock = threading.Lock()
frames_served = [0]

@app.get("/info")
def info():
    return jsonify(ok=True, neurons=int(brain.n), links=int(cn.W.nnz), dataset=DATASET if 'TOKEN' in globals() and TOKEN else "DEMO",
                   eye=int(len(brain.vis)), giant_fiber=int(len(brain.dn)),
                   sample_types=[str(cn.types[i]) for i in ctrl.samp], sample_n=int(len(ctrl.samp)), frames=frames_served[0])

@app.post("/frame")
def frame():
    m = request.get_json(force=True, silent=True) or {}
    with _lock:
        frames_served[0] += 1
        return jsonify(ctrl.frame(m.get("dist"), bool(m.get("on_ground", True))))

@app.post("/reset")
def reset():
    with _lock: ctrl.reset()
    return jsonify(ok=True)

print("brain + API ready")

In [ ]:
#@title 4) Start the API + public tunnel (copy the printed URL into the game)
import os, re, subprocess, threading, time, urllib.request
from werkzeug.serving import make_server

PORT = 5000
if globals().get("_srv") is not None:
    try: _srv.shutdown()
    except Exception: pass
_srv = make_server("0.0.0.0", PORT, app, threaded=True)
threading.Thread(target=_srv.serve_forever, daemon=True).start()
print(f"API running locally on port {PORT}")

if globals().get("_tun") is not None:
    try: _tun.kill()
    except Exception: pass
bin_path = "/content/cloudflared"
if not os.path.exists(bin_path):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", bin_path)
    os.chmod(bin_path, 0o755)
_tun = subprocess.Popen([bin_path, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None; t0 = time.time()
while time.time() - t0 < 40 and PUBLIC_URL is None:
    line = _tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m: PUBLIC_URL = m.group(0)
if PUBLIC_URL:
    print("\n==============================================")
    print("  API URL  (paste into the game page):")
    print("  ", PUBLIC_URL)
    print("==============================================")
    print("Keep this Colab tab open while you play. Test link:", PUBLIC_URL + "/info")
else:
    print("Tunnel URL not found. Re-run this cell. (Fallback: use ngrok with your own token.)")

**Stop / restart:** to change settings, edit cell 2, then re-run cells 2, 3 and 4 (a new URL is printed). Keep this tab open while playing.